A notebook used for exploring the data after fine-tuning Level 1 (Aircraft v. Rest)

# Setup

In [ ]:
import datasets
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download

cache_path = "/content/huggingface_cache"
os.makedirs(cache_path, exist_ok=True)
os.environ['HF_HOME'] = cache_path

if userdata.get('HF_TOKEN'):
  login(token=userdata.get('HF_TOKEN'))
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="utils.py", repo_type="dataset",local_dir=".")

raw_dataset = datasets.load_dataset("sookiemonster/asrs-narratives")

utils.py: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.74M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/9.38M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10360 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4441 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9868 [00:00<?, ? examples/s]

In [ ]:
train_ds = raw_dataset['train']
valid_ds = raw_dataset['validation']
test_ds = raw_dataset['test']

labels = train_ds.features['label'].names

id_to_label = { idx : label for idx, label in enumerate(labels) }
label_to_id = { label : idx for idx, label in id_to_label.items() }

In [ ]:
from functools import partial

def filter_labels(ds, to_remove:list):
  to_remove_set = set(to_remove)
  return ds.filter(lambda example : id_to_label[example['label']] not in to_remove_set)

filter_ambiguous = partial(filter_labels, to_remove=['ambiguous'])

filtered_train_ds = filter_ambiguous(train_ds)
filtered_valid_ds = filter_ambiguous(valid_ds)

Filter:   0%|          | 0/10360 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4441 [00:00<?, ? examples/s]

In [ ]:
from datasets import ClassLabel

groupings = {
    'all' : set(id_to_label.values())
    # 'ac' : set(['aircraft']),
    # 'not-ac' : set(id_to_label.values()) - set(['aircraft'])
}

def _validate_groupings(groupings:dict[str, set]):
  mut_excl = [va.isdisjoint(vb) for ka, va in groupings.items() for kb, vb in groupings.items() if ka != kb]
  assert all(mut_excl)

  all_labels = set([label for val_set in groupings.values() for label in val_set])
  assert all_labels == set(id_to_label.values()), f"Missing: {set(id_to_label.values()) - all_labels}"


def group_labels(ds, groupings:dict[str, set]):
  _validate_groupings(groupings)
  group_names = list(groupings.keys())
  group_names.sort()

  fine_grained_label_to_group = {
      label : group_name for group_name, val_set in groupings.items() for label in val_set
  }

  res = ds.map(lambda ex: {"group" : fine_grained_label_to_group[ id_to_label[ex['label']] ]})

  new_features = res.features.copy()
  new_features["group"] = ClassLabel(names=group_names)

  res = res.cast(new_features)
  return res

grouped_ds_train = group_labels(filtered_train_ds, groupings)
grouped_ds_valid = group_labels(filtered_valid_ds, groupings)

Map:   0%|          | 0/9525 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/9525 [00:00<?, ? examples/s]

Map:   0%|          | 0/4083 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/4083 [00:00<?, ? examples/s]

# Get Embeddings

In [ ]:
import torch
import sys

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}")
print(f"PyTorch Version: {torch.__version__.split('+')[0]}")
print(f"CUDA Version (Torch): {torch.version.cuda}")
print(f"CXX11 ABI: {torch._C._GLIBCXX_USE_CXX11_ABI}")

In [ ]:
# !pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "sookiemonster/asrs-modernbert-aircraft-vs-rest"

def preprocess(ds:datasets):
    PREFIX = "classification: "
    res = ds.map(lambda examples : {"text" : PREFIX + examples["text"]})
    return res

pre_train = preprocess(filtered_train_ds)
pre_valid = preprocess(filtered_valid_ds)

Map:   0%|          | 0/9525 [00:00<?, ? examples/s]

Map:   0%|          | 0/4083 [00:00<?, ? examples/s]

In [ ]:
id_to_group = { i: group for i, group in enumerate(pre_train.features['label'].names) }
group_to_id = { group: i for i, group in id_to_group.items()}

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_ID, device="cuda", model_kwargs={"torch_dtype" : torch.float16})

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: sookiemonster/asrs-modernbert-aircraft-vs-rest
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Validation Embeddings

In [ ]:
embeddings = model.encode(pre_valid['text'], show_progress_bar=True, batch_size=8)
print(embeddings.shape)

In [ ]:
np.save("tc_aircraft_versus_rest_embeddings_drop_classification_head.npy", embeddings)

## Training Embeddings

In [ ]:
t_embeddings = model.encode(pre_train['text'], show_progress_bar=True, batch_size=8)
print(t_embeddings.shape)

Batches:   0%|          | 0/1191 [00:00<?, ?it/s]

(9525, 768)


In [ ]:
np.save("tc_aircraft_versus_rest_embeddings_drop_classification_head_training.npy", t_embeddings)

# Run UMAP on Validation Set
Want to see which classes seem to be separable.

In [ ]:
tembed = np.load("tc_aircraft_versus_rest_embeddings_drop_classification_head_training.npy")
embed = np.load("tc_aircraft_versus_rest_embeddings_drop_classification_head.npy")

## 3d

In [ ]:
from umap import UMAP

um = UMAP(n_neighbors=10, min_dist=0.1, metric='cosine', n_components=3)
proj = um.fit_transform(embed)

In [ ]:
import plotly.express as px

fig = px.scatter_3d(x=proj[:,0], y=proj[:,1], z=proj[:,2],
              color=pd.Series(filtered_valid_ds['label']).map(id_to_label))
fig.show()

## 2d

In [ ]:
from umap import UMAP

um = UMAP(n_neighbors=10, min_dist=0.1, metric='cosine', n_components=2)
proj = um.fit_transform(embed)

In [ ]:
import plotly.express as px

fig = px.scatter(x=proj[:,0], y=proj[:,1],
              color=pd.Series(filtered_valid_ds['label']).map(id_to_label))
fig.show()

## Matryoshka Truncation

In [ ]:
mat_embed = embed[:, :256]
mat_tembed = tembed[:, :256]
mat_embed.shape

(4083, 256)

In [ ]:
train_y = color=pd.Series(filtered_train_ds['label']).map(id_to_label)
valid_y = color=pd.Series(filtered_valid_ds['label']).map(id_to_label)

In [ ]:
from umap import UMAP

um = UMAP(n_neighbors=10, min_dist=0.1, metric='cosine', n_components=2)
proj = um.fit_transform(mat_embed)

In [ ]:
import plotly.express as px

fig = px.scatter(x=proj[:,0], y=proj[:,1],
              color=pd.Series(filtered_valid_ds['label']).map(id_to_label))
fig.show()